In [ ]:
# Cell 1 - Clone repository
%cd /content
!if [ ! -d VDT_GD2 ]; then git clone https://github.com/Darkin72/VDT_GD2.git; fi
%cd /content/VDT_GD2
!git pull


In [ ]:
# Cell 2 - Paper training configuration
DEVICE = "cuda"
MODE = "ITS"  # ITS: I-HAZE + O-HAZY + SOTS-ITS; OTS: SOTS-OTS only
WEIGHT_DECAY = 1e-4
MIN_LR = 1e-6
PATIENCE = 50
MULTI_SCALE = (64, 128, 256)
AUGMENTATION = "flip H/V, rotation 0/+90/-90, brightness/contrast +/-0.1"

CONFIG = {
    # Paper: synthetic datasets use 1000 epochs, batch 16 and cosine annealing.
    # ITS settings follow Table 1 of the local paper (Section 4.2 prose gives a different LR).
    "SOTS_ITS": {"epochs": 1000, "batch": 16, "lr": 2e-4, "scheduler": "cosine"},
    "SOTS_OTS": {"epochs": 1000, "batch": 16, "lr": 1e-4, "scheduler": "cosine"},
    # I-HAZE/O-HAZY are real-world datasets; use the paper's real-world settings.
    "I_HAZE": {"epochs": 500, "batch": 8, "lr": 2e-4, "scheduler": "step"},
    "O_HAZY": {"epochs": 500, "batch": 8, "lr": 2e-4, "scheduler": "step"},
}
for dataset, cfg in CONFIG.items():
    print(dataset, cfg)
print("mode:", MODE, "| device:", DEVICE, "| weight_decay:", WEIGHT_DECAY, "| min_lr:", MIN_LR)
print("multi-scale:", MULTI_SCALE, "| augmentation:", AUGMENTATION)

EVAL_MAX_SIDE = 1024  # Match Bench.ipynb-style bounded full-image evaluation

import os
import torch

NUM_WORKERS = min(4, os.cpu_count() or 1)
PREFETCH_FACTOR = 2
MICRO_BATCH_SIZE = 16
USE_AMP = True
AMP_DTYPE = "bfloat16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else "float16"
PROFILE_BATCHES = 0
USE_MULTI_SCALE = True

print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "capability:", torch.cuda.get_device_capability(0))
print("workers:", NUM_WORKERS, "prefetch:", PREFETCH_FACTOR, "micro batch limit:", MICRO_BATCH_SIZE, "AMP:", USE_AMP, AMP_DTYPE)



In [ ]:
# Cell 3 - Download data and run the complete four-dataset baseline
!pip -q install pillow matplotlib numpy kagglehub tqdm openpyxl psutil scikit-image thop pandas
%cd /content/VDT_GD2

from pathlib import Path
import os
import subprocess
import sys
import kagglehub

if MODE not in ("ITS", "OTS"):
    raise ValueError("MODE must be ITS or OTS")

its_root = Path(kagglehub.dataset_download("balraj98/indoor-training-set-its-residestandard")) if MODE == "ITS" else Path("dataset")
ots_root = Path(kagglehub.dataset_download("brunobelloni/outdoor-training-set-ots-reside")) if MODE == "OTS" else Path("dataset")
sots_download = Path(kagglehub.dataset_download("balraj98/synthetic-objective-testing-set-sots-reside"))
print("MODE:", MODE)
print("ITS:", its_root)
print("OTS:", ots_root)
print("SOTS:", sots_download)

i_haze_train = Path("dataset/I-HAZE/train").resolve()
o_hazy_train = Path("dataset/O-HAZY/train").resolve()
if MODE == "ITS":
    for path in (i_haze_train, o_hazy_train):
        if not (path / "hazy").is_dir() or not (path / "clear").is_dir():
            raise FileNotFoundError(f"Missing paired training data: {path}/hazy and {path}/clear")

sots_root = Path("dataset/Synthetic Objective Testing Set (SOTS) [RESIDE]")
for domain in ("indoor", "outdoor"):
    for kind in ("clear", "hazy"):
        target = sots_root / domain / "test" / kind
        target.parent.mkdir(parents=True, exist_ok=True)
        source = sots_download / domain / kind
        if not source.is_dir():
            raise FileNotFoundError(source)
        if not target.exists():
            target.symlink_to(source, target_is_directory=True)

def paired_root(path):
    path = Path(path)
    if (path / "hazy").is_dir() and (path / "clear").is_dir():
        return path
    candidates = sorted(parent.parent for parent in path.rglob("hazy") if (parent.parent / "clear").is_dir())
    if len(candidates) != 1:
        raise ValueError(f"Cannot find one paired root under {path}: {candidates}")
    return candidates[0]

if MODE == "ITS":
    i_haze_train = paired_root(i_haze_train)
    o_hazy_train = paired_root(o_hazy_train)
else:
    i_haze_train = Path("dataset")
    o_hazy_train = Path("dataset")
output_dir = Path(f"outputs/hazewavenet_{MODE.lower()}")
output_dir.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, "-u", "-m", "solution.wavelet_dehaze.run_full_pipeline",
    "--i-haze-data", str(i_haze_train), "--o-hazy-data", str(o_hazy_train),
    "--its-data", str(its_root), "--ots-data", str(ots_root), "--mode", MODE,
    "--eval-data-root", "dataset", "--output-dir", str(output_dir),
    "--device", DEVICE, "--size", "256", "--micro-batch-size", str(MICRO_BATCH_SIZE),
    "--weight-decay", str(WEIGHT_DECAY), "--min-lr", str(MIN_LR), "--patience", str(PATIENCE),
    "--val-fraction", "0.1", "--num-workers", str(NUM_WORKERS), "--prefetch-factor", str(PREFETCH_FACTOR),
    "--profile-repeats", "10", "--amp-dtype", AMP_DTYPE,
    "--augment", "--multi-scale", "--pin-memory", "--persistent-workers", "--amp",
]
print("Running complete four-dataset baseline:")
print(" ".join(command), flush=True)
env = {**os.environ, "PYTHONUNBUFFERED": "1", "HW_TQDM": "0"}
print("Launching child pipeline now; live logs follow:", flush=True)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, command)
print("Completed:", output_dir.resolve(), flush=True)




In [ ]:
# Cell 4 - Plot and inspect all four training histories
import json
from pathlib import Path
import matplotlib.pyplot as plt

output_dir = Path(f"outputs/hazewavenet_{MODE.lower()}")
files = {"I-HAZE": output_dir / "history_i_haze.json", "O-HAZY": output_dir / "history_o_hazy.json", "SOTS-ITS": output_dir / "history_sots_its.json", "SOTS-OTS": output_dir / "history_sots_ots.json"}
histories = {name: json.loads(path.read_text()) for name, path in files.items() if path.exists()}
expected = {"I-HAZE", "O-HAZY", "SOTS-ITS"} if MODE == "ITS" else {"SOTS-OTS"}
if set(histories) != expected:
    raise FileNotFoundError(f"Expected histories {expected} under {output_dir}: {list(histories)}")
fig, axes = plt.subplots(len(histories), 3, figsize=(20, 5 * len(histories)), squeeze=False)
for row, (name, records) in enumerate(histories.items()):
    epochs = [item["epoch"] for item in records]
    for col, (train_key, val_key, title) in enumerate((("train_loss", "val_loss", "Loss"), ("train_psnr", "val_psnr", "PSNR (dB)"), ("train_ssim", "val_ssim", "SSIM"))):
        ax = axes[row, col]
        ax.plot(epochs, [item[train_key] for item in records], label="Train")
        ax.plot(epochs, [item[val_key] for item in records], "--", label="Validation")
        ax.set_title(f"{name} - {title}"); ax.set_xlabel("Epoch"); ax.set_ylabel(title); ax.grid(alpha=0.25); ax.legend()
plt.tight_layout(); fig.savefig(output_dir / "training_metrics_all_datasets.png", dpi=160); plt.show()
for name, records in histories.items():
    best = max(records, key=lambda item: item.get("val_psnr", float("-inf")))
    print(name, "best_val_psnr=", best.get("val_psnr"), "epoch=", best["epoch"])


In [ ]:
# Cell 5 - List checkpoints, inference images and reports
from pathlib import Path
output_dir = Path(f"outputs/hazewavenet_{MODE.lower()}")
for subdir in (output_dir, output_dir / "inference", output_dir / "reports"):
    print(f"\n{subdir}")
    for path in sorted(subdir.rglob("*")) if subdir.exists() else []:
        if path.is_file(): print(" -", path)


In [ ]:
# Cell 6 - Detailed stage timing (preprocess + model decomposition/branches)
import time
import numpy as np
import torch
from PIL import Image
from solution.wavelet_dehaze.model import HazeWaveNet

TIMING_REPEATS = 10
WARMUP = 3
TIMING_SIZE = 256  # Match training input; set to 1024 to profile bounded evaluation.

def timed_preprocess(path):
    start = time.perf_counter()
    with Image.open(path) as image:
        image = image.convert("RGB").resize((TIMING_SIZE, TIMING_SIZE), Image.Resampling.BILINEAR)
    tensor = torch.from_numpy(np.array(image, copy=True)).float().div(255).permute(2, 0, 1).unsqueeze(0)
    return tensor, (time.perf_counter() - start) * 1000

def profile_dataset(label, checkpoint, folder):
    model = HazeWaveNet().to(DEVICE).eval()
    state = torch.load(checkpoint, map_location=DEVICE)
    model.load_state_dict(state.get("model", state))
    paths = sorted(p for p in Path(folder).iterdir() if p.is_file())
    if not paths:
        raise FileNotFoundError(folder)
    sample, _ = timed_preprocess(paths[0]); sample = sample.to(DEVICE)
    for _ in range(WARMUP): model(sample)
    rows = []
    for path in paths:
        image, preprocess_ms = timed_preprocess(path)
        image = image.to(DEVICE)
        _, stage = model.profile_forward(image, repeats=TIMING_REPEATS)
        stage["preprocess"] = preprocess_ms
        stage["total_with_preprocess"] = preprocess_ms + stage["total"]
        rows.append(stage)
    keys = ["preprocess", "dwt_mfde_decomposition", "dcp_fixed_prior", "low_frequency_fegg_fegb", "high_frequency_vp_vr", "wim_reconstruction", "final_refinement", "total", "total_with_preprocess"]
    print(f"\n{label} ({len(rows)} images, size={TIMING_SIZE}x{TIMING_SIZE}, device={DEVICE})")
    for key in keys:
        print(f"{key:32s}: {np.mean([row[key] for row in rows]):9.3f} ms")

profile_dataset("I-HAZE", "haze_wavelet_i_haze.pt", "dataset/I-HAZE/test/hazy")
profile_dataset("O-HAZY", "haze_wavelet_o_hazy.pt", "dataset/O-HAZY/test/hazy")
